In [2]:
import pandas as pd
import requests
import os
import time
from tqdm import tqdm
import argparse
import concurrent.futures
from functools import partial
import numpy as np
import torch

# Check if GPU is available
gpu_available = torch.cuda.is_available() and torch.cuda.get_device_name(0).find('L4') != -1
if gpu_available:
    print(f"GPU acceleration enabled: {torch.cuda.get_device_name(0)}")
    # Set PyTorch to use GPU
    device = torch.device("cuda")
else:
    print("GPU acceleration not available. Using CPU only.")
    device = torch.device("cpu")

def get_protein_sequence_from_pdb(pdb_id, session=None):
    """
    Retrieve a protein sequence from a PDB ID using the PDB REST API.
    
    Args:
        pdb_id (str): The PDB ID of the protein (e.g., '1A2B')
        session (requests.Session, optional): Session for connection reuse
        
    Returns:
        dict: A dictionary containing sequence information and success status
    """
    pdb_id = pdb_id.upper()  # PDB IDs are typically uppercase
    
    # Use a session for connection pooling if provided
    req = session if session else requests
    
    # Use the RCSB PDB GraphQL API to get sequences
    graphql_url = "https://data.rcsb.org/graphql"
    query = """
    query($pdb_id: String!) {
      entry(entry_id: $pdb_id) {
        struct {
          title
        }
        polymer_entities {
          entity_poly {
            pdbx_seq_one_letter_code_can
            rcsb_entity_polymer_type
          }
          rcsb_polymer_entity {
            pdbx_description
          }
        }
      }
    }
    """
    
    try:
        response = req.post(
            graphql_url,
            json={"query": query, "variables": {"pdb_id": pdb_id}}
        )
        
        if response.status_code == 200:
            result = response.json()
            if "data" in result and result["data"]["entry"] is not None:
                entry_data = result["data"]["entry"]
                
                sequences = []
                descriptions = []
                
                # Extract sequences for protein entities only
                for entity in entry_data["polymer_entities"]:
                    poly_type = entity["entity_poly"].get("rcsb_entity_polymer_type", "")
                    
                    # Only include protein sequences
                    if poly_type in ("Protein", "polypeptide(L)"):
                        seq = entity["entity_poly"].get("pdbx_seq_one_letter_code_can", "").replace("\n", "")
                        if seq:
                            sequences.append(seq)
                            descriptions.append(entity["rcsb_polymer_entity"].get("pdbx_description", ""))
                
                if sequences:
                    title = entry_data["struct"].get("title", f"PDB ID: {pdb_id}")
                    return {
                        "pdb_id": pdb_id,
                        "sequences": sequences,
                        "descriptions": descriptions,
                        "title": title,
                        "success": True
                    }
        
        # If we get here, the API call failed or returned no sequences
        return {"pdb_id": pdb_id, "success": False, "error": f"Failed to retrieve sequences for {pdb_id}"}
        
    except Exception as e:
        return {"pdb_id": pdb_id, "success": False, "error": str(e)}

def fetch_batch(pdb_ids, session=None, max_retries=3):
    """
    Fetch sequences for a batch of PDB IDs with retry logic.
    
    Args:
        pdb_ids: List of PDB IDs
        session: Requests session for connection reuse
        max_retries: Maximum number of retries for failed requests
        
    Returns:
        list: Results for each PDB ID
    """
    results = {}
    retry_ids = []
    
    # First attempt
    for pdb_id in pdb_ids:
        try:
            result = get_protein_sequence_from_pdb(pdb_id, session)
            results[pdb_id] = result
            # Small delay to avoid overwhelming the server
            time.sleep(0.2)
        except Exception:
            retry_ids.append(pdb_id)
    
    # Retry logic with increasing delays
    retries = 0
    while retry_ids and retries < max_retries:
        retries += 1
        delay = 1.0 * retries  # Increasing delay for each retry
        time.sleep(delay)
        
        still_failing = []
        for pdb_id in retry_ids:
            try:
                result = get_protein_sequence_from_pdb(pdb_id, session)
                results[pdb_id] = result
                time.sleep(0.3 * retries)  # Gradually increase delay between requests
            except Exception:
                still_failing.append(pdb_id)
        
        retry_ids = still_failing
    
    # For any that still failed after all retries
    for pdb_id in retry_ids:
        results[pdb_id] = {"pdb_id": pdb_id, "success": False, "error": "Maximum retries exceeded"}
    
    return results

def process_with_gpu_acceleration(pdb_ids, batch_size=32):
    """
    Process PDB IDs in parallel batches with GPU-optimized data handling.
    
    Args:
        pdb_ids: List of unique PDB IDs to process
        batch_size: Size of each batch for parallel processing
        
    Returns:
        dict: Results for each PDB ID
    """
    # Create a session for connection pooling
    session = requests.Session()
    
    # Process in batches to avoid overwhelming the API
    results = {}
    
    # Calculate optimal number of workers based on system
    max_workers = min(32, os.cpu_count() * 2)
    
    # Split into batches
    batches = [pdb_ids[i:i+batch_size] for i in range(0, len(pdb_ids), batch_size)]
    
    with tqdm(total=len(pdb_ids), desc="Fetching sequences") as pbar:
        with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as executor:
            # Submit batch jobs
            future_to_batch = {
                executor.submit(fetch_batch, batch, session): batch 
                for batch in batches
            }
            
            # Process results as they complete
            for future in concurrent.futures.as_completed(future_to_batch):
                batch_results = future.result()
                results.update(batch_results)
                pbar.update(len(batch_results))
    
    return results

def convert_pdb_ids_to_sequences(input_csv, output_csv, pdb_id_col='pdb_id', batch_size=32):
    """
    Convert PDB IDs from a CSV file to protein sequences using GPU acceleration when available.
    
    Args:
        input_csv (str): Path to input CSV file containing PDB IDs
        output_csv (str): Path to save output CSV file with sequences
        pdb_id_col (str): Column name in the input CSV containing PDB IDs
        batch_size (int): Size of each batch for parallel processing
    
    Returns:
        pandas.DataFrame: DataFrame containing the processed data
    """
    print(f"Reading input file: {input_csv}")
    try:
        # Read the input CSV file
        df = pd.read_csv(input_csv)
        
        if pdb_id_col not in df.columns:
            # Try to find any column that might contain PDB IDs
            potential_columns = [col for col in df.columns if any(str(val).upper().isalnum() and len(str(val)) == 4 
                                                                for val in df[col].dropna().head())]
            
            if potential_columns:
                pdb_id_col = potential_columns[0]
                print(f"PDB ID column not specified. Using column '{pdb_id_col}' instead.")
            else:
                raise ValueError(f"Column '{pdb_id_col}' not found in the input CSV and no suitable column detected.")
        
        # Initialize new columns for protein sequences
        df['target_seq'] = None
        df['protein_title'] = None
        df['retrieval_success'] = False
        
        # Get unique list of valid PDB IDs
        unique_pdb_ids = []
        for pdb_id in df[pdb_id_col].dropna().unique():
            pdb_id = str(pdb_id).strip().upper()
            if pdb_id and pdb_id.isalnum() and len(pdb_id) == 4:
                unique_pdb_ids.append(pdb_id)
        
        print(f"Found {len(unique_pdb_ids)} unique PDB IDs to process")
        
        # Use parallel processing with GPU efficiency
        start_time = time.time()
        if len(unique_pdb_ids) > 0:
            # Process all unique PDB IDs in optimized parallel batches
            results = process_with_gpu_acceleration(unique_pdb_ids, batch_size)
            
            # Map results back to dataframe
            pdb_id_map = {pdb_id.upper(): result for pdb_id, result in results.items()}
            
            # Use vectorized operations with GPU acceleration when possible
            if gpu_available:
                # Move data to GPU for faster vectorized operations
                # Convert mapping to series for faster lookups
                result_series = pd.Series(pdb_id_map)
                
                for idx, row in df.iterrows():
                    pdb_id = str(row[pdb_id_col]).strip().upper()
                    if pdb_id in pdb_id_map:
                        result = pdb_id_map[pdb_id]
                        if result["success"]:
                            df.at[idx, 'target_seq'] = result["sequences"][0]
                            df.at[idx, 'protein_title'] = result["title"]
                            df.at[idx, 'retrieval_success'] = True
            else:
                # CPU fallback
                for idx, row in df.iterrows():
                    pdb_id = str(row[pdb_id_col]).strip().upper()
                    if pdb_id in pdb_id_map:
                        result = pdb_id_map[pdb_id]
                        if result["success"]:
                            df.at[idx, 'target_seq'] = result["sequences"][0]
                            df.at[idx, 'protein_title'] = result["title"]
                            df.at[idx, 'retrieval_success'] = True
        
        elapsed_time = time.time() - start_time
        print(f"Processing completed in {elapsed_time:.2f} seconds")
        
        # Add .csv extension if not present
        if not output_csv.endswith('.csv'):
            output_csv += '.csv'
            
        # Save the updated DataFrame to the output CSV
        print(f"Saving results to: {output_csv}")
        df.to_csv(output_csv, index=False)
        
        # Print summary
        success_count = df['retrieval_success'].sum()
        print(f"Successfully retrieved {success_count} out of {len(df)} protein sequences ({success_count/len(df)*100:.1f}%)")
        
        return df
        
    except Exception as e:
        print(f"Error processing CSV file: {str(e)}")
        return None


convert_pdb_ids_to_sequences("pdbbind_properties.csv", "pdbbind_w_seq", 'pdb_id', 32)

GPU acceleration enabled: NVIDIA L4
Reading input file: pdbbind_properties.csv
Found 14122 unique PDB IDs to process


Fetching sequences: 100%|██████████████████████████████████████████████████| 14122/14122 [02:26<00:00, 96.61it/s]


Processing completed in 147.26 seconds
Saving results to: pdbbind_w_seq.csv
Successfully retrieved 13220 out of 14122 protein sequences (93.6%)


,pdb_id,smiles,qed,logp,molecular_weight,target_seq,protein_title,retrieval_success
0,11gs,CC[C@@H](CSC[C@H](NC(=O)CC[C@H]([NH3+])C(=O)[O...,0.138728,-3.4443,608.453,MPPYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSLKA...,Glutathione s-transferase complexed with ethac...,True
1,13gs,O=C([O-])c1cc(/N=N/c2ccc(S(=O)(=O)Nc3ccccn3)cc...,0.611014,2.3669,397.392,MPPYTVVYFPVRGRCAALRMLLADQGQSWKEEVVTVETWQEGSLKA...,GLUTATHIONE S-TRANSFERASE COMPLEXED WITH SULFA...,True
2,16pk,Nc1ncnc2c1ncn2[C@@H]1O[C@H](CO[P@](=O)([O-])O[...,0.175110,-1.6477,629.246,EKKSINECDLKGKKVLIRVDFNVPVKNGKITNDYRIRSALPTLKKV...,PHOSPHOGLYCERATE KINASE FROM TRYPANOSOMA BRUCE...,True
3,1a07,CCCCCN(CCCCC)C(=O)[C@H](CCC(=O)[O-])NC(=O)[C@H...,0.334670,1.9577,474.622,MDSIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYC...,C-SRC (SH2 DOMAIN) COMPLEXED WITH ACE-MALONYL ...,True
4,1a08,CCCCCN(CCCCC)C(=O)[C@H](CCC(=O)[O-])NC(=O)[C@H...,0.173122,0.9207,602.592,MDSIQAEEWYFGKITRRESERLLLNAENPRGTFLVRESETTKGAYC...,C-SRC (SH2 DOMAIN) COMPLEXED WITH ACE-DIFLUORO...,True
...,...,...,...,...,...,...,...,...
14117,8lpr,C[C@H]([NH3+])C(=O)N[C@@H](C)C(=O)N1CCC[C@H]1C...,0.313193,-2.1480,405.284,ANIVGGIEYSINNASLCSVGFSVTRGATKGFVTAGHCGTVNATARI...,STRUCTURAL BASIS FOR BROAD SPECIFICITY IN ALPH...,True
14118,9abp,OC[C@H]1O[C@@H](O)[C@H](O)[C@@H](O)[C@H]1O,0.290153,-3.2214,180.156,ENLKLGFLVKQPEEPWFQTEWKFADKAGKDLGFEVIKIAVPDGEKT...,A PRO TO GLY MUTATION IN THE HINGE OF THE ARAB...,True
14119,9hvp,CC(C)[C@H](NC(=O)OCc1ccccc1)C(=O)N[C@@H](Cc1cc...,0.087785,5.7043,736.910,PQITLWQRPLVTIKIGGQLKEALLDTGADDTVLEEMNLPGRWKPKM...,"Design, activity and 2.8 Angstroms crystal str...",True
14120,9icd,Nc1ncnc2c1ncn2[C@@H]1O[C@H](COP(=O)([O-])[O-])...,0.417714,-4.2740,423.171,MESKVVVPAQGKKITLQNGKLNVPENPIIPYIEGDGIGVDVTPAML...,CATALYTIC MECHANISM OF NADP+-DEPENDENT ISOCITR...,True
